# AutoML fuer Zeitreihen mit AutoGluon

Dieses Notebook adressiert den AutoML-Teil der Agenda. Es zeigt einen kompakten Forecasting-Workflow mit mehreren Serien, bei dem AutoGluon Modellwahl und Ensembling automatisiert.

## Lernziele

- ein Panel-Dataset fuer Zeitreihen aufbauen
- die AutoGluon TimeSeries-API kennenlernen
- ein AutoML-Forecasting-Setup trainieren
- Leaderboard und Forecast-Ergebnisse interpretieren


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor
except ImportError as exc:
    raise ImportError('Dieses Notebook benoetigt autogluon.timeseries. Installiere es mit `pip install autogluon.timeseries pyarrow`.') from exc

np.random.seed(42)


## 1. Panel-Datensatz mit mehreren Serien erzeugen

Statt nur einer Reihe modellieren wir mehrere Produkte. Das ist eine typische AutoML-Situation, weil globale Modelle und Ensembling ueber mehrere Serien hinweg besonders sinnvoll werden.


In [ ]:
dates = pd.date_range('2022-01-01', periods=180, freq='D')
items = [f'product_{i}' for i in range(1, 7)]
rows = []
for item_idx, item in enumerate(items):
    base = 20 + item_idx * 2
    trend = np.linspace(0, 3 + item_idx * 0.3, len(dates))
    weekly = 2.5 * np.sin(2 * np.pi * np.arange(len(dates)) / 7 + item_idx / 3)
    monthly = 1.2 * np.cos(2 * np.pi * np.arange(len(dates)) / 30)
    noise = np.random.normal(0, 0.6, size=len(dates))
    target = base + trend + weekly + monthly + noise
    for ts, y in zip(dates, target):
        rows.append({'item_id': item, 'timestamp': ts, 'target': float(y)})

panel_df = pd.DataFrame(rows)
panel_df.head()


In [ ]:
sample_items = panel_df['item_id'].unique()[:3]
fig, ax = plt.subplots(figsize=(13, 4))
for item in sample_items:
    subset = panel_df[panel_df['item_id'] == item]
    ax.plot(subset['timestamp'], subset['target'], label=item)
ax.set_title('Beispielhafte Panel-Zeitreihen')
ax.set_xlabel('Datum')
ax.set_ylabel('Zielwert')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## 2. Train/Test-Split fuer Forecasting


In [ ]:
PREDICTION_LENGTH = 14

train_parts = []
test_parts = []
for item in items:
    subset = panel_df[panel_df['item_id'] == item].sort_values('timestamp')
    train_parts.append(subset.iloc[:-PREDICTION_LENGTH])
    test_parts.append(subset)

train_df = pd.concat(train_parts, ignore_index=True)
test_df = pd.concat(test_parts, ignore_index=True)

train_data = TimeSeriesDataFrame.from_data_frame(train_df, id_column='item_id', timestamp_column='timestamp')
test_data = TimeSeriesDataFrame.from_data_frame(test_df, id_column='item_id', timestamp_column='timestamp')
train_data.head()


## 3. AutoGluon Predictor trainieren

Das Preset `fast_training` ist fuer Trainingsumgebungen auf CPU ein guter Start. Fuer laengere Workshops oder echte Daten koennen spaeter staerkere Presets ausprobiert werden.


In [ ]:
predictor = TimeSeriesPredictor(
    prediction_length=PREDICTION_LENGTH,
    target='target',
    eval_metric='MASE',
)

predictor.fit(
    train_data=train_data,
    presets='fast_training',
    time_limit=90,
)


In [ ]:
leaderboard = predictor.leaderboard(train_data, silent=True)
leaderboard


In [ ]:
evaluation = predictor.evaluate(test_data, silent=True)
evaluation


## 4. Forecasts erzeugen und visualisieren


In [ ]:
predictions = predictor.predict(train_data)
predictions.head()


In [ ]:
item = items[0]
history = panel_df[panel_df['item_id'] == item].sort_values('timestamp')
history_train = history.iloc[:-PREDICTION_LENGTH]
history_true_future = history.iloc[-PREDICTION_LENGTH:]
pred_item = predictions.loc[item].reset_index()

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(history_train['timestamp'], history_train['target'], label='train_history', linewidth=2)
ax.plot(history_true_future['timestamp'], history_true_future['target'], label='true_future', linewidth=2)
ax.plot(pred_item['timestamp'], pred_item['mean'], label='autogluon_forecast', linewidth=2, linestyle='--')
if '0.1' in pred_item.columns and '0.9' in pred_item.columns:
    ax.fill_between(pred_item['timestamp'], pred_item['0.1'], pred_item['0.9'], alpha=0.2, label='80% interval')
ax.set_title(f'Forecast fuer {item}')
ax.set_xlabel('Datum')
ax.set_ylabel('Zielwert')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## Fazit

- AutoML ist besonders stark, wenn schnell mehrere Modellklassen verglichen werden sollen.
- AutoGluon verbindet klassische und tiefe Modelle unter einer API.
- Fuer echte Projekte bleiben Datenqualitaet, Forecast-Horizont und fachliche Plausibilitaet entscheidend.

## Uebungen

1. Vergleiche `fast_training` mit einem staerkeren Preset.
2. Erhoehe `PREDICTION_LENGTH` auf 28 Tage.
3. Ersetze die synthetischen Panel-Daten durch einen realen Demand- oder Sensor-Datensatz.
